# Inuktitut Q&A — Fine-Tuning Pipeline
**Model Training & Comparison**

This notebook covers:
1. Environment setup (Unsloth + LoRA + 4-bit quantization)
2. Data loading from JSONL
3. Base model baseline — run **all 48 test questions**, save outputs
4. LoRA fine-tuning on 194 training samples
5. Adapted model — run same 48 questions, save outputs
6. Keyword-based accuracy scoring per topic
7. Side-by-side comparison output saved to file
8. Save adapter for UI backend

---

## Step 0 — Check GPU
Run this first. If it says No GPU, go to Runtime > Change runtime type > T4 GPU.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime > Change runtime type > T4 GPU')

## Step 1 — Install Dependencies
> This takes 3-5 minutes the first time. Run once, do not re-run.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets
print('Done.')

## Step 2 — Upload Data Files

Upload both files:
- `train_qa_pairs.jsonl` — 194 samples, what the model **learns from**
- `test_qa_pairs.jsonl` — 48 samples, the **exam** (model never sees these during training)

In [ ]:
from google.colab import files
print('Please upload: train_qa_pairs.jsonl and test_qa_pairs.jsonl')
uploaded = files.upload()

In [ ]:
import json
from collections import Counter

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_data = load_jsonl('train_qa_pairs.jsonl')
test_data  = load_jsonl('test_qa_pairs.jsonl')

print(f'Training samples : {len(train_data)}  <- model learns from these')
print(f'Test samples     : {len(test_data)}   <- model is evaluated on these')
print()
print('Test questions by topic:')
for topic, count in Counter(q['context'] for q in test_data).items():
    print(f'  {topic:<15} {count} questions')

## Step 3 — Load Base Model (Qwen2.5-3B-Instruct, 4-bit)

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT   = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'Qwen/Qwen2.5-3B-Instruct',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
)
print('Base model loaded.')

## Step 4 — Define Prompt Template

In [ ]:
SYSTEM_PROMPT = 'You are a helpful assistant specialized in Inuktitut culture and communities.'

def format_prompt(instruction, context, response=''):
    prompt = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{instruction}\n\nContext: {context}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    if response:
        prompt += f'{response}<|im_end|>'
    return prompt

print('Prompt template ready. Example:')
print(format_prompt(train_data[0]['instruction'], train_data[0]['context'], train_data[0]['response']))

## Step 5 — Helper Function to Generate Answers

In [ ]:
def generate_response(mdl, tkn, instruction, context, max_new_tokens=256):
    FastLanguageModel.for_inference(mdl)
    prompt = format_prompt(instruction, context)
    inputs = tkn(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = 0.1,
            do_sample      = True,
            pad_token_id   = tkn.eos_token_id,
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tkn.decode(new_tokens, skip_special_tokens=True).strip()

## Step 6 — Run Base Model on ALL 48 Test Questions
These are the before fine-tuning answers. Estimated time: ~8 minutes on T4.

In [ ]:
print(f'Running base model on all {len(test_data)} test questions...\n')
base_outputs = []

for i, q in enumerate(test_data):
    answer = generate_response(model, tokenizer, q['instruction'], q['context'])
    base_outputs.append(answer)
    if (i + 1) % 10 == 0 or i == 0:
        print(f'  [{i+1}/{len(test_data)}] {q["instruction"][:60]}')
        print(f'    -> {answer[:100]}\n')

print('Base model outputs collected.')

## Step 7 — Apply LoRA for Fine-Tuning

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 32,
    target_modules = ['q_proj', 'v_proj', 'k_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params (LoRA): {trainable:,}  ({100*trainable/total:.2f}% of total)')
print(f'Frozen params (base)   : {total-trainable:,}')

## Step 8 — Prepare Training Dataset

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list([
    {'text': format_prompt(item['instruction'], item['context'], item['response'])}
    for item in train_data
])

print(f'Training dataset ready: {len(train_dataset)} examples')

## Step 9 — Train (Fine-Tune)
**Expected time on free Colab T4: ~10-12 minutes**

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_dataset,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_ratio        = 0.1,
        num_train_epochs    = 5,
        learning_rate       = 1e-4,
        fp16                = not is_bfloat16_supported(),
        bf16                = is_bfloat16_supported(),
        logging_steps       = 10,
        optim               = 'adamw_8bit',
        weight_decay        = 0.01,
        lr_scheduler_type   = 'cosine',
        output_dir          = './checkpoints',
        seed                = 42,
    ),
)

print('Starting fine-tuning... (~10 mins on T4)')
stats = trainer.train()
print(f'Training complete!')
print(f'  Time      : {stats.metrics["train_runtime"]:.0f} seconds')
print(f'  Final loss: {stats.metrics["train_loss"]:.4f}  (lower = better, aim for < 1.0)')

## Step 10 — Run Adapted Model on Same 48 Test Questions

In [ ]:
print(f'Running adapted model on all {len(test_data)} test questions...\n')
adapted_outputs = []

for i, q in enumerate(test_data):
    answer = generate_response(model, tokenizer, q['instruction'], q['context'])
    adapted_outputs.append(answer)
    if (i + 1) % 10 == 0 or i == 0:
        print(f'  [{i+1}/{len(test_data)}] {q["instruction"][:60]}')
        print(f'    -> {answer[:100]}\n')

print('Adapted model outputs collected.')

## Step 11 — Keyword-Based Accuracy Scoring

For each question we know the correct answer (from `test_qa_pairs.jsonl`). We check how many key words from the correct answer appear in each model's response. This scores both models automatically across all 48 questions and all 6 topics.

In [ ]:
def keyword_score(model_answer, reference_answer):
    STOPWORDS = {'the','a','an','is','in','of','and','to','it','for','or',
                 'that','this','was','are','as','by','with','its','at','be',
                 'from','has','have','on'}
    ref_words   = set(w.lower().strip('.,;:') for w in reference_answer.split()) - STOPWORDS
    model_words = set(w.lower().strip('.,;:') for w in model_answer.split())
    if not ref_words:
        return 0.0
    return len(ref_words & model_words) / len(ref_words)

results = []
for i, q in enumerate(test_data):
    base_s    = keyword_score(base_outputs[i],    q['response'])
    adapted_s = keyword_score(adapted_outputs[i], q['response'])
    results.append({
        'question'    : q['instruction'],
        'topic'       : q['context'],
        'reference'   : q['response'],
        'base_ans'    : base_outputs[i],
        'adapted_ans' : adapted_outputs[i],
        'base_score'  : base_s,
        'adapted_score': adapted_s,
        'improved'    : adapted_s > base_s,
    })

avg_base    = sum(r['base_score']    for r in results) / len(results)
avg_adapted = sum(r['adapted_score'] for r in results) / len(results)
n_improved  = sum(1 for r in results if r['improved'])

print('=' * 55)
print('OVERALL SCORES (all 48 questions)')
print('=' * 55)
print(f'  Base model avg score    : {avg_base:.2%}')
print(f'  Adapted model avg score : {avg_adapted:.2%}')
print(f'  Questions improved      : {n_improved} / {len(results)}')
print()
print('SCORES BY TOPIC')
print('-' * 55)
topics = sorted(set(r['topic'] for r in results))
for topic in topics:
    tr = [r for r in results if r['topic'] == topic]
    tb = sum(r['base_score']    for r in tr) / len(tr)
    ta = sum(r['adapted_score'] for r in tr) / len(tr)
    arrow = 'UP' if ta > tb else ('DOWN' if ta < tb else 'SAME')
    print(f'  {topic:<15} base={tb:.2%}  adapted={ta:.2%}  [{arrow}]  (n={len(tr)})')

## Step 12 — Save Full Side-by-Side Comparison to File

In [ ]:
import textwrap

SEP  = '=' * 80
DSEP = '-' * 80
lines = []

lines += [
    SEP,
    'INUKTITUT Q&A -- BASE vs ADAPTED MODEL COMPARISON',
    f'Model: Qwen2.5-3B-Instruct | LoRA r=16 | Trained on {len(train_data)} samples',
    f'Evaluated on: ALL {len(test_data)} test questions across 6 topics',
    SEP, '',
    'SUMMARY', DSEP,
    f'  Base model avg keyword score    : {avg_base:.2%}',
    f'  Adapted model avg keyword score : {avg_adapted:.2%}',
    f'  Questions where adapted model improved: {n_improved} / {len(results)}',
    '', '  SCORES BY TOPIC:',
]
for topic in topics:
    tr = [r for r in results if r['topic'] == topic]
    tb = sum(r['base_score']    for r in tr) / len(tr)
    ta = sum(r['adapted_score'] for r in tr) / len(tr)
    lines.append(f'    {topic:<15} base={tb:.2%}  adapted={ta:.2%}  (n={len(tr)})')
lines += ['', SEP, '']

for i, r in enumerate(results):
    lines += [
        f'Q{i+1} [{r["topic"].upper()}]: {r["question"]}',
        DSEP,
        'REFERENCE ANSWER:',
        textwrap.fill(r['reference'], 78), '',
        f'BASE MODEL (score: {r["base_score"]:.2%}):',
        textwrap.fill(r['base_ans'], 78), '',
        f'ADAPTED MODEL (score: {r["adapted_score"]:.2%}) -- {"IMPROVED" if r["improved"] else "same or lower"}:',
        textwrap.fill(r['adapted_ans'], 78), '',
        SEP, '',
    ]

with open('comparison_base_vs_adapted.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print(f'Saved to comparison_base_vs_adapted.txt  ({len(lines)} lines, {len(results)} questions)')

## Step 13 — Save LoRA Adapter

In [ ]:
import os
ADAPTER_PATH = 'inuktitut_lora_adapter'
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f'Adapter saved to ./{ADAPTER_PATH}/')
for fname in os.listdir(ADAPTER_PATH):
    size = os.path.getsize(os.path.join(ADAPTER_PATH, fname))
    print(f'  {fname:<40} {size/1e6:.1f} MB')

## Step 14 — Download Everything

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('inuktitut_lora_adapter', 'zip', ADAPTER_PATH)
files.download('inuktitut_lora_adapter.zip')      
files.download('comparison_base_vs_adapted.txt')
print('Downloads started.')